# Sesión 17 · Ventanas para señales y secuencias

Badge 3 · Cálculo avanzado con NumPy

Calcula indicadores móviles y distingue una ventana de observaciones de una ponderación de señal.

150 minutos de clase y 45–60 minutos de tarea. Ejecuta en orden.

## Ventanas deslizantes sin ciclos explícitos

sliding_window_view crea una vista de ventanas superpuestas. Para N observaciones y ancho W, obtienes N−W+1 ventanas completas. Cada fila contiene W observaciones contiguas.

La vista comparte memoria y es de solo lectura por defecto. Evita escribir en ventanas superpuestas. Aunque crear la vista puede ser barato, reducir todas las ventanas cuesta aproximadamente O(N×W).

In [1]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
x=np.array([10.,20.,30.,40.,50.])
w=sliding_window_view(x,3)
print(w)
print(w.mean(axis=-1))

[[10. 20. 30.]
 [20. 30. 40.]
 [30. 40. 50.]]
[20. 30. 40.]


**Qué observar:** Ventanas [10,20,30], [20,30,40], [30,40,50]; medias [20,30,40].

**Recuerda:** Estas ventanas usan posiciones, no fechas. Si el tiempo es irregular, vuelve a definir la frecuencia o usa Pandas.

## Pesos, convolución y significado de window

Una media ponderada necesita pesos con suma positiva; normalízalos para sumar uno. Aquí el peso mayor corresponde a la observación más reciente y la salida se alinea al final de cada ventana.

np.convolve con mode="valid" produce solo posiciones con solapamiento completo. Para pesos asimétricos, la convolución invierte el núcleo. Las ventanas Hann o Hamming son ponderaciones de señal, no operaciones rolling por sí solas.

In [2]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
x=np.array([10.,20.,30.,40.,50.])
pesos=np.array([1.,2.,3.]); pesos=pesos/pesos.sum()
y=sliding_window_view(x,3)@pesos
print(np.round(y,2))
print(np.allclose(y,np.convolve(x,pesos[::-1],mode="valid")))
print(np.hanning(5))

[23.33 33.33 43.33]
True
[0.  0.5 1.  0.5 0. ]


**Qué observar:** Medias [23.33,33.33,43.33]; equivalencia True. Hann de 5: [0,0.5,1,0.5,0].

**Recuerda:** No confundas una función de ponderación con una ventana de observaciones: pueden combinarse, pero cumplen papeles distintos.

## Bordes, alineación y cálculo eficiente

Una media móvil de ancho 3 no está definida para las primeras dos posiciones si exiges una ventana completa. Puedes alinear las salidas con el final de cada ventana y conservar NaN al principio.

Para datos finitos, una suma acumulada permite calcular sumas móviles en O(N). Con NaN este atajo requiere sumas y conteos separados. Para predecir el valor actual, desplaza antes: una ventana que incluye el presente no es una variable exclusivamente pasada.

In [3]:
import numpy as np
x=np.array([10.,20.,30.,40.,50.])
k=3
s=np.r_[0.,np.cumsum(x)]
medias=(s[k:]-s[:-k])/k
alineada=np.r_[np.full(k-1,np.nan),medias]
print(alineada)

[nan nan 20. 30. 40.]


**Qué observar:** [NaN,NaN,20,30,40].

**Recuerda:** Documenta ancho, mínimo de válidos, alineación y política de bordes antes de comparar resultados.

## Reto · Media de ancho dos

Calcula ventanas completas de [2,4,6,8] con ancho 2.

**Criterio:** [3,5,7].

In [ ]:
# Tu solución aquí.

## Reto · Pesos con prioridad reciente

Usa pesos [1,3] normalizados sobre [2,4,6,8].

**Criterio:** [3.5,5.5,7.5].

In [ ]:
# Tu solución aquí.

## Tarea · Laboratorio · Window functions

Calcula indicadores móviles y distingue una ventana de observaciones de una ponderación de señal.

1. Genera 30 observaciones con semilla fija y una anomalía intencional.
2. Compara medias móviles simples y ponderadas con anchos 3 y 5.
3. Verifica una media con suma acumulada; documenta bordes y alineación.
4. Integra una máscara de inválidos e informa conteos mínimos y limitaciones.

In [ ]:
# Desarrolla tu tarea y explica los resultados.